# Video sampling logic

In [1]:
import pickle
import torch
from pathlib import Path
import numpy as np
import random
import pandas as pd

In [2]:
project_root = Path.cwd().parent
data_root = str(project_root / "multisports_data/data/trainval")
ground_truth_path= str(project_root / "multisports_fewshot_GT.pkl")

# Read The GT file
with open(ground_truth_path, "rb") as file:
    data = pickle.load(file)

# GTtubes
gttubes = data["gttubes"]

# Get the test classes
possible_classes = data["test_labels"]

# Splitting logic to query and support 
# Pick N novel classes
N = 5
# Pick at least M samples from those classes
M = 15
# NUmber of support samples
K = 5

# Sample classes
novel_classes_ids = random.sample(range(len(possible_classes)), N)

# Test videos
test_videos = data["test_videos"]

# For now do so that it is >= M, for results you can always only take M 

# Get the videos that include those classes and sample at least K 
video_dict = {} # video id and number of instances of the action in the class action : {video_id1: 4, video_id2: 4}


def get_m_vids_from_novel_class(samples_set, novel_classes_ids, M):

    for novel_class_id in novel_classes_ids:
        video_dict[novel_class_id] = {}

        # Go through all the videos in test videos
        for video in samples_set:
            tubes = gttubes[video]
            # Count the number of instances in the video
            if novel_class_id in gttubes[video].keys():
                instance_count = len(tubes[novel_class_id])
                video_dict[novel_class_id][video] = instance_count

    # Now sample those classes in order (they are already random so it's fine if we don't mix again)
    query_videos = set()
    accumulated_counts = {cls: 0 for cls in novel_classes_ids}

    for cls, dct in video_dict.items():

        # Shuffle the videos so that we pick random instances every time 
        video_items = list(dct.items())
        random.shuffle(video_items)

        for video, cnt in video_items:
            # Stop when enough instances 
            if accumulated_counts[cls] >= M:
                break
            # if not yet in query videos add it to the query videos list
            if video not in query_videos:
                query_videos.add(video)

                # Add the counts for all of the classes that are in the video
                for any_cls in novel_classes_ids:
                    if video in video_dict[any_cls]:
                        accumulated_counts[any_cls] += video_dict[any_cls][video]
                

    return list(query_videos), accumulated_counts


query_videos, accumulated_counts = get_m_vids_from_novel_class(test_videos, novel_classes_ids, M)


# Count the number of instances we go
for cls, cnts in accumulated_counts.items():
    print(f"{cnts} instances for class: {cls}")

# Pick K instances for the prototype - support set, from the remaining videos
remaining_vids = list(set(test_videos) - set(query_videos))
support_videos, accumulated_counts = get_m_vids_from_novel_class(remaining_vids, novel_classes_ids, K)

print()
# Count the number of instances we go
for cls, cnts in accumulated_counts.items():
    print(f"{cnts} instances for class: {cls}")


# Random order of classes - than start by sampling videos for first class - for the next ones already count the videos decided
# If there is too many for some class, just do not count those ones or something like that 
# You have to exclude them randomly
        
# print(gttubes)  

16 instances for class: 0
17 instances for class: 6
20 instances for class: 3
15 instances for class: 4
16 instances for class: 14

5 instances for class: 0
5 instances for class: 6
8 instances for class: 3
5 instances for class: 4
5 instances for class: 14


# The actual evaluation

In [7]:
from ultralytics import YOLO
from decord import VideoReader
from decord import cpu, gpu
from PIL import Image
from tqdm import tqdm
import torchvision.transforms as T

detection_model = YOLO("yolo26n.pt")

In [8]:
def load_video(path, F = 8):
    vr = VideoReader(path, ctx=cpu(0))
    frames =  vr.get_batch(range(len(vr)))
    del vr
    return frames.asnumpy()


In [ ]:
# Evaluation loop
E = 1

device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
model.eval()
model = model.to(device)

transform = T.Compose([
    T.Resize((224, 224), antialias=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


for episode in tqdm(range(E)):
    # Sample the query and support videos
    query_videos, _ = get_m_vids_from_novel_class(test_videos, novel_classes_ids, M)
    remaining_vids = list(set(test_videos) - set(query_videos))
    support_videos, _ = get_m_vids_from_novel_class(remaining_vids, novel_classes_ids, K)

    # Build prototypes
    prototypes = {id:[] for id in novel_classes_ids}

    for id in novel_classes_ids:
        for vid in support_videos:
            path = data_root + f"/{vid}.mp4"
            frames = load_video(path)
            frames_tensor = torch.from_numpy(frames).permute(0, 3, 1, 2).float()
            frames_tensor = frames_tensor / 255.0

            frames_tensor = transform(frames_tensor).to(device)

            with torch.no_grad():
                frame_features = model(frames_tensor)

            video_prototype = frame_features.mean(dim=0)

            prototypes[id].append(video_prototype.cpu())
            

Using cache found in C:\Users\marko/.cache\torch\hub\facebookresearch_dinov2_main
  0%|          | 0/1 [00:03<?, ?it/s]


RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same